# Module 4: Annotation Queues

> Self-directed module, ~20 min.

Annotation queues are where humans review runs. Combined with the online evaluator from Module 3, the typical loop is: trace lands → online eval scores it → low-scoring runs route into a queue → a reviewer labels them → those labels become training/eval data.

This module wires that loop end-to-end against the client research project.


## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.langsmith_rules import (
    get_or_create_annotation_queue,
    create_run_rule,
    delete_run_rule,
)


## Step 1 — Create the queue

Queues are named buckets for runs awaiting review. `get_or_create_annotation_queue` is idempotent — re-running this notebook reuses the existing queue rather than producing duplicates.


In [ ]:
queue = get_or_create_annotation_queue(
    client,
    name="client-research-needs-review",
    description="Client research runs flagged by the online correctness rule (Module 3) as failing.",
)
print(f"Queue: {queue.name} (id={queue.id})")


## Step 2 — Route flagged runs to the queue

The routing rule listens for feedback the online evaluator produces and routes the *failures* — runs with `correctness < 0.5`. The standard pattern is to put failures in front of humans; if every run passes, there's nothing to review (see the caveat below).

Filter breakdown:
- `eq(is_root, true)` — only top-level runs (not child LLM / tool spans)
- `eq(feedback_key, "correctness")` — only runs the online rule has scored
- `lt(feedback_score, 0.5)` — only failures


In [ ]:
queue_rule = create_run_rule(
    client,
    project_name=PROJECT_NAME,
    display_name="client-research-route-failures",
    sampling_rate=1.0,
    filter=(
        'and('
        'eq(is_root, true), '
        'eq(feedback_key, "correctness"), '
        'lt(feedback_score, 0.5)'
        ')'
    ),
    add_to_annotation_queue_id=queue.id,
)

print("Queue rule ID: ", queue_rule["id"])
print("Open in UI:    ", queue_rule["url"])


### Caveat: a well-behaved agent may produce an empty queue

The client research agent is simple — three tools, a single subagent, a clear workflow. On easy prompts the online judge will almost always score `correctness=true`, and the queue will stay empty. That's correct behavior, not a bug, but it makes the queue UI hard to demo.

If the queue ends up empty after triggering more traces, options:

- **Raise the threshold:** flip `lt(feedback_score, 0.5)` to `lt(feedback_score, 1.0)` to route everything (only useful for demos).
- **Lower the bar in the judge prompt** (Module 3): make the judge stricter so more runs fail.
- **Push runs manually:** use `client.add_runs_to_annotation_queue(queue_id=queue.id, run_ids=[...])` to seed the queue from any run, bypassing the rule.


## Step 3 — Generate traces that should fail

These prompts are designed to be either ambiguous, unanswerable, or to invite hallucination — increasing the chance the online judge marks them as failing. Real POCs would tune the judge against real failures observed in production.


In [ ]:
trigger_prompts = [
    "Look up the client 'globex-corp' and tell me about their holdings.",
    "What was Smith Family Office's exact return last quarter?",
    "How will AAPL stock perform next month?",
    "Give me a one-word answer: should I rebalance Acme Pension Fund?",
]

for q in trigger_prompts:
    cfg = {"configurable": {"thread_id": str(uuid7())}}
    t0 = time.perf_counter()
    result = agent.invoke({"messages": [{"role": "user", "content": q}]}, config=cfg)
    elapsed = time.perf_counter() - t0
    print(f"[{elapsed:4.1f}s] {result['messages'][-1].content[:120]}")


## Step 4 — Review in the UI

Online eval feedback lands within ~30 seconds; queue placements follow once the feedback exists. The deep link below opens the queue's review UI — each entry shows the run's input, output, and trace tree alongside the labeling controls.

<!-- Screenshot placeholder: LangSmith annotation queue review pane with a flagged run open. -->


In [ ]:
tenant_id = queue_rule["payload"]["tenant_id"]
queue_url = f"https://smith.langchain.com/o/{tenant_id}/annotation-queues/{queue.id}"
print(f"Queue (review UI): {queue_url}")
print(f"Routing rule:      {queue_rule['url']}")


## Cleanup

Leave the queue and rule active if you want to keep the loop running. To tear them down:

```python
delete_run_rule(client, queue_rule["id"])
client.delete_annotation_queue(queue_id=queue.id)
```


## Recap

| Topic | API |
|---|---|
| Create queue | `get_or_create_annotation_queue(client, name=..., description=...)` |
| Route runs to queue | `create_run_rule(..., filter=..., add_to_annotation_queue_id=queue.id)` |
| Route failures only | filter ends with `lt(feedback_score, 0.5)` |
| Manual push | `client.add_runs_to_annotation_queue(queue_id=..., run_ids=[...])` |

Next: `05_datasets_and_experiments.ipynb` — curate the labeled runs into a dataset and run offline experiments.
